# Kuramoto Oscillators on a Ring

Nearest-neighbor Kuramoto model on a 1D periodic chain. Run all cells, then scroll to the bottom to play with `K`, `N`, and `seed` interactively.

Source: [velvetmonkey/flywheel-universe](https://github.com/velvetmonkey/flywheel-universe)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from ipywidgets import interact, FloatSlider, IntSlider


In [ ]:
def kuramoto(t, theta, omega, K, N):
    """RHS of Kuramoto ODE on a ring (nearest-neighbor coupling)."""
    dtheta = np.zeros(N)
    for i in range(N):
        ip = (i + 1) % N
        im = (i - 1) % N
        dtheta[i] = omega[i] + K * (np.sin(theta[ip] - theta[i]) + np.sin(theta[im] - theta[i]))
    return dtheta


def order_parameter(theta):
    """Magnitude of the complex order parameter r."""
    z = np.mean(np.exp(1j * theta))
    return np.abs(z)


In [ ]:
N = 120
np.random.seed(42)
omega = np.random.normal(0.0, 0.65, N)
t_span = (0, 150)


## 1. Bifurcation diagram — order parameter vs coupling strength

In [ ]:
K_values = np.linspace(0.0, 4.0, 35)
r_final = []
for K in K_values:
    theta0 = np.random.uniform(0, 2*np.pi, N)
    sol = solve_ivp(kuramoto, t_span, theta0, args=(omega, K, N), method='RK45', rtol=1e-6, atol=1e-8)
    r_final.append(order_parameter(sol.y[:, -1]))

plt.figure(figsize=(9, 5))
plt.plot(K_values, r_final, 'o-', linewidth=2, markersize=4)
plt.axvline(x=1.2, color='red', linestyle='--', alpha=0.3, label='Mean-field K_c (does not apply here)')
plt.xlabel('Coupling strength K')
plt.ylabel('Order parameter r')
plt.title('Bifurcation Diagram — Kuramoto Oscillators on a Ring')
plt.grid(True, alpha=0.5)
plt.legend()
plt.tight_layout()


## 2. Supercritical simulation (K = 2.5) — strong synchronization

In [ ]:
K_super = 2.5
theta0 = np.random.uniform(0, 2*np.pi, N)
sol = solve_ivp(kuramoto, t_span, theta0, args=(omega, K_super, N),
                method='RK45', dense_output=True, rtol=1e-6, atol=1e-8)
t_plot = np.linspace(0, 80, 1200)

R_real, R_imag, r_time = [], [], []
for t in t_plot:
    z = np.mean(np.exp(1j * sol.sol(t)))
    R_real.append(z.real)
    R_imag.append(z.imag)
    r_time.append(np.abs(z))
R_real = np.array(R_real)
R_imag = np.array(R_imag)


### A. Time evolution of the order parameter

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(t_plot, r_time, linewidth=2.5)
plt.xlabel('Time')
plt.ylabel('Order parameter r(t)')
plt.title(f'Time evolution of synchronization (K = {K_super})')
plt.grid(True, alpha=0.5)
plt.ylim(0, 1.05)
plt.tight_layout()


### B. Phase portrait in the complex plane

In [ ]:
plt.figure(figsize=(7, 7))
plt.plot(R_real, R_imag, 'b-', linewidth=1.8, alpha=0.85, label='Trajectory')
plt.plot(R_real[0], R_imag[0], 'go', markersize=8, label='Initial (incoherent)')
plt.plot(R_real[-1], R_imag[-1], 'ro', markersize=8, label='Final (chimera/twisted)')
plt.xlabel('Re(R)')
plt.ylabel('Im(R)')
plt.title('Phase Portrait of the Order Parameter\nQuasi-periodic orbit — not a spiral to sync')
plt.axis('equal')
plt.grid(True, alpha=0.4)
plt.legend()
plt.tight_layout()


### C. Phase evolution heatmap

In [ ]:
theta_sol = np.array([sol.sol(t) for t in t_plot]).T
phases_wrapped = theta_sol % (2 * np.pi)

plt.figure(figsize=(11, 6))
plt.pcolormesh(t_plot, np.arange(N), phases_wrapped, cmap='twilight_shifted', shading='auto')
plt.colorbar(label='Phase θ_i (mod 2π)')
plt.xlabel('Time')
plt.ylabel('Oscillator index i (along the ring)')
plt.title('Phase Evolution on the Ring')
plt.tight_layout()


### D. Initial vs final ring configuration

In [ ]:
def plot_ring(theta, title, ax):
    x = np.cos(theta)
    y = np.sin(theta)
    ax.scatter(x, y, c=theta, cmap='twilight_shifted', s=60, edgecolor='black', linewidth=0.5)
    for i in range(len(theta)):
        ax.plot([x[i], x[(i+1) % len(theta)]], [y[i], y[(i+1) % len(theta)]], 'k-', alpha=0.25, linewidth=1)
    ax.set_title(title)
    ax.axis('equal')
    ax.set_xticks([])
    ax.set_yticks([])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))
plot_ring(theta0, 'Initial: Incoherent', ax1)
plot_ring(sol.sol(t_plot[-1]), f'Final ring state (K={K_super}) — twisted/chimera, not global sync', ax2)
plt.suptitle('Oscillators on a Ring — winding number selection and chimera states')
plt.tight_layout()


## 3. Play with it

Drag the sliders. Each change re-seeds, re-integrates, and re-renders the phase-evolution heatmap and the final ring configuration.

- **K** — coupling strength. Below ~1.2 → incoherence. Above → synchronization.
- **N** — number of oscillators on the ring.
- **seed** — randomness for natural frequencies and initial phases.


In [ ]:
@interact(
    K=FloatSlider(min=0.0, max=4.0, step=0.1, value=2.5, description='K'),
    N=IntSlider(min=20, max=200, step=10, value=120, description='N'),
    seed=IntSlider(min=0, max=100, step=1, value=42, description='seed'),
)
def explore(K, N, seed):
    np.random.seed(seed)
    omega = np.random.normal(0.0, 0.65, N)
    theta0 = np.random.uniform(0, 2*np.pi, N)
    sol = solve_ivp(kuramoto, (0, 80), theta0, args=(omega, K, N),
                    method='RK45', dense_output=True, rtol=1e-6, atol=1e-8)
    t_plot = np.linspace(0, 80, 600)
    theta_sol = np.array([sol.sol(t) for t in t_plot]).T
    phases_wrapped = theta_sol % (2 * np.pi)
    theta_final = sol.sol(t_plot[-1])

    fig, (axL, axR) = plt.subplots(1, 2, figsize=(14, 5))
    pc = axL.pcolormesh(t_plot, np.arange(N), phases_wrapped, cmap='twilight_shifted', shading='auto')
    fig.colorbar(pc, ax=axL, label='Phase (mod 2π)')
    axL.set_xlabel('Time')
    axL.set_ylabel('Oscillator index')
    axL.set_title(f'Phase evolution  (K={K:.2f}, N={N}, seed={seed})')

    x = np.cos(theta_final)
    y = np.sin(theta_final)
    axR.scatter(x, y, c=theta_final, cmap='twilight_shifted', s=60, edgecolor='black', linewidth=0.5)
    for i in range(N):
        axR.plot([x[i], x[(i+1) % N]], [y[i], y[(i+1) % N]], 'k-', alpha=0.25, linewidth=1)
    axR.set_title(f'Final ring  (r = {order_parameter(theta_final):.3f})')
    axR.axis('equal')
    axR.set_xticks([])
    axR.set_yticks([])
    plt.tight_layout()
    plt.show()
